In [1]:
cd ..

/Users/camila.cusicanqui/Documents/GitHub/frod-agentic-ai


In [32]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [33]:
import os
import yaml

# Load credentials into environment variables so database, OpenAI, and
# Google client libraries can reuse the same local secrets as the rest of the repo.
with open(".config/credentials-mage.yaml", "r") as f:
    creds = yaml.safe_load(f)

# The credentials file stores name/value pairs; exporting them here keeps the
# notebook cells below free of hard-coded secret values.
for item in creds:
    os.environ[item["name"]] = str(item["value"])

In [34]:
import importlib.util
import sys
from pathlib import Path

TRANSACTION_SCHEMA_PATH = Path(
    "/Users/camila.cusicanqui/Documents/GitHub/analytics-mage-infra/mage-fraud-space/fraud_utils/fraud_agents/schemas/transactions.py"
)
spec = importlib.util.spec_from_file_location("fraud_transaction_schemas", TRANSACTION_SCHEMA_PATH)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not load transaction schemas from {TRANSACTION_SCHEMA_PATH}")

transaction_schemas = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = transaction_schemas
spec.loader.exec_module(transaction_schemas)

TransactionRow = transaction_schemas.TransactionRow
TransactionBatch = transaction_schemas.TransactionBatch

In [35]:
# homemade utils
from utils.data_ingest import get_db_conn
from utils.chargeback_data import load_cb_df
# librerías 
import pandas as pd
import numpy as np
import gspread
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, timedelta, datetime
import gspread
from dateutil.utils import today

# formatting options
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:,.2f}'.format


In [6]:
# cb_df = pd.read_excel('/Users/camila.cusicanqui/Documents/klar/mini-tasks/misc_info/trx_alerts_trthfdr.xlsx')

In [7]:
cb_query = """
select
    t.*,
    tc.cb_timestamp,
    tc.cb_reason,
    tc.mcc_code
from
    ops_fraud.card_purchase_transactions_enriched_31d t
inner join
    ops_fraud.total_chargeback tc on t.transaction_id = tc.transaction_id
-- where t.product_type = 'PLATINUM'
"""

In [8]:
cb_df = pd.read_sql(cb_query, get_db_conn())

/var/folders/ch/hb63cwpx22l_1jl5cqj7wp_80000gq/T/ipykernel_676/3630719773.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  cb_df = pd.read_sql(cb_query, get_db_conn())


In [9]:

CB_REASON_MAP = {
    "Transacción por internet o compra telefónica": "Rembolso no recibido",
    "Error durante el proceso": "Cargo no reconocido",
    "Otro": "Otro",
    "Producto no recibido": "Rembolso no recibido",
    "Transacción duplicada": "Transacción duplicada",
    "Transacción declinada": "Transacción declinada",
    "Transacción excede el importe autorizado": "Rembolso no procesado",
    "Aclaración de Transacción de Cajero Automático": "Aclaración de Transacción de Cajero Automático",
    "ATM no entrega dinero": "Aclaración de Transacción de Cajero Automático",
    "Reporte de cargo no reconocido": "Cargo no reconocido",
    "TRANSACTION_EXCEEDS_AUTHORIZED_AMOUNT": "Rembolso no procesado",
    "Reembolso no procesado": "Rembolso no procesado",
    "REFUND_NOT_PROCESSED": "Rembolso no procesado",
    "TRANSACTION_DECLINED": "Transacción declinada",
    "OTHER": "Otro",
    "UNRECOGNIZED_TRANSACTION": "Cargo no reconocido",
    "INTERNET_PHONE_TRANSACTION": "Rembolso no recibido",
    "UNRECOGNIZED_CHARGE": "Cargo no reconocido",
    "DUPLICATE_TRANSACTION": "Transacción duplicada",
    "ATM_TRANSACTION_CLARIFICATION": "Aclaración de Transacción de Cajero Automático",
}


ONLINE_POS_ENTRY_MODES = {"CNP Manual", "CNP Card on File"}


In [10]:
cb_df = cb_df.drop_duplicates(subset=["transaction_id"], keep="first")
cb_df = cb_df[
    (cb_df["transaction_id"].notna())
    & (cb_df["amount_abs"].notna())
    & (cb_df["cb_reason"] != "Devolución de dinero")
].copy()

cb_df["online"] = cb_df["pos_entry_mode"].isin(ONLINE_POS_ENTRY_MODES)
cb_df["cb_reason_original"] = cb_df["cb_reason"]
cb_df["cb_reason"] = cb_df["cb_reason"].map(CB_REASON_MAP)
cb_df["Regla de fraude"] = None
cb_df["embozo_file_date"] = pd.to_datetime(
    cb_df["nombrearchivo"].str.extract(r"par(\d{6})", expand=False),
    format="%y%m%d",
    errors="coerce"
)

In [11]:
cb_df.rename(columns={
    'timestamp_mx_created_at': 'trx_timestamp_mx',
    'amount_abs': 'amount_pos',
    'cvv_flag': 'cvv_ind',
    'pin_capabilities': 'metodo_identificacion',
    'validation_3ds': 'three_ds_status',
    't3ds_indicator': 'three_ds_flow',
    'acquirer_id': 'adquirente',
    'merchant_country_code': 'country',
    'response_code': 'cod_respuesta'
    }, inplace=True)

In [12]:
# Keep this list aligned with the column dictionary in system_prompt.txt.
# The notebook only exposes columns that actually exist in cb_df, so the same
# prompt can run across slightly different extracts without failing on optional fields.
requested_transaction_cols = [
    'Regla de fraude',
    'cb_timestamp',
    'user_id',
    'klrid',
    'transaction_id',
    'trx_timestamp_mx',
    'amount_pos',
    'operador',
    'mcc_code',
    'card_type',
    'product_type',
    'nombrearchivo',
    'embozo_file_date',
    'pos_entry_mode',
    'online',
    'cvv_ind',
    'metodo_identificacion',
    'three_ds_status',
    'three_ds_flow',
    'afiliacion',
    'adquirente',
    'country',
    'cod_respuesta',
]

# important_cols is the exact dataframe surface area exposed to the agent tools.
# Missing columns are reported for transparency but are not fatal.
important_cols = [column for column in requested_transaction_cols if column in cb_df.columns]
missing_transaction_cols = [column for column in requested_transaction_cols if column not in cb_df.columns]

print(f"Using {len(important_cols)} transaction columns")
if missing_transaction_cols:
    print("Missing optional columns:", missing_transaction_cols)


Using 23 transaction columns


In [13]:
cb_df.groupby(['three_ds_status','three_ds_flow']).transaction_id.nunique()

three_ds_status  three_ds_flow
Challenge        3ds               157
No 3ds           No 3ds           3508
No challenge     3ds                96
no_data          no_data           719
Name: transaction_id, dtype: int64

In [36]:
import asyncio
#from agents import Agent, Runner
#from agents import Agent, Runner, function_tool
import datetime as dt
import polars as pl
# import docx2txt
from pydantic import BaseModel, Field
from typing import List, Optional
import json


In [15]:
# Convert dataframe values into JSON-safe prompt values for the small preview prompt below.
# This is only for inspection; the actual experiment runner uses dataframe tools instead.
def _transaction_value_for_prompt(column, value):
    if pd.isna(value):
        return None
    if column == "amount_pos":
        return float(value)
    if hasattr(value, "isoformat"):
        return value.isoformat(sep=" ")
    return str(value)


def build_transaction_batch(df: pd.DataFrame, limit: int | None = None) -> TransactionBatch:
    # Limit is useful when manually inspecting the prompt payload without dumping the full dataframe.
    rows_df = df.loc[:, important_cols]
    if limit is not None:
        rows_df = rows_df.head(limit)

    records = [
        {
            column: _transaction_value_for_prompt(column, value)
            for column, value in row.items()
        }
        for row in rows_df.to_dict("records")
    ]
    return TransactionBatch(transactions=records)



In [16]:

transactions_for_prompt = build_transaction_batch(cb_df, limit=50)
transaction_context = transactions_for_prompt.model_dump_json(indent=2)
user_prompt = f"""Describe the transactional patterns in this chargeback batch.

Transactions:
{transaction_context}
"""

user_prompt

'Describe the transactional patterns in this chargeback batch.\n\nTransactions:\n{\n  "transactions": [\n    {\n      "cb_timestamp": "2026-05-18 18:36:45.677038",\n      "user_id": "c888e402-0ca0-47f1-a510-8dc7bc0d8d08",\n      "transaction_id": "231149626",\n      "trx_timestamp_mx": "2026-05-16 00:25:19.290000",\n      "amount_pos": 70.0,\n      "operador": "DLO*UBER RIDE         CIUDAD DE MEXNA MX",\n      "mcc_code": "4121",\n      "card_type": "VIRTUAL",\n      "product_type": "CREDIT_5401_BIN",\n      "nombrearchivo": null,\n      "pos_entry_mode": "CNP Manual",\n      "cvv_ind": "no_cvv",\n      "metodo_identificacion": "unknown",\n      "three_ds_flow": "No 3ds",\n      "afiliacion": "2000001",\n      "adquirente": "418.0",\n      "country": "MX",\n      "cod_respuesta": "0.0"\n    },\n    {\n      "cb_timestamp": "2026-05-27 04:52:53.983580",\n      "user_id": "5fd6f3ee-9c73-4e8e-a578-c887c9b2c102",\n      "transaction_id": "231150831",\n      "trx_timestamp_mx": "2026-05-16 

## Local Python transaction analyst agent

These cells expose `cb_df[important_cols]` to the agent through controlled local Python tools instead of dumping the full dataframe into the prompt.

In [37]:
from types import SimpleNamespace
import importlib
import transactional_describer.transaction_agent as transaction_agent

# Force-refresh the local module after editing transaction_agent.py while this
# notebook kernel is already running.
transaction_agent = importlib.reload(transaction_agent)

from transactional_describer.transaction_agent import (
    TransactionAnomalyReport,
    TransactionalDescription,
    build_transaction_analyst_agent,
    build_transaction_anomaly_agent,
    find_transaction_anomalies,
    get_transaction_schema,
    prepare_transaction_context,
    run_transaction_python,
    screen_transaction_anomalies,
)

In [18]:
# Prepare the local tool context once for a quick smoke test of the agent tools.
# The dataframe is persisted as JSONL and read by controlled Python tools, which avoids
# putting the full dataset directly in the model prompt.
transaction_agent_context = prepare_transaction_context(
    cb_df[cb_df['trx_timestamp_mx'] >= '2026-05-25'],
    important_cols,
    sample_name="chargeback_transactions",
)

print(f"Saved agent dataset to: {transaction_agent_context.data_path}")
print(f"Columns exposed: {transaction_agent_context.columns}")

Saved agent dataset to: /var/folders/ch/hb63cwpx22l_1jl5cqj7wp_80000gq/T/chargeback_transactions_320220821e4a4090b87331a8da8143a8.jsonl
Columns exposed: ['Regla de fraude', 'cb_timestamp', 'user_id', 'klrid', 'transaction_id', 'trx_timestamp_mx', 'amount_pos', 'operador', 'mcc_code', 'card_type', 'product_type', 'nombrearchivo', 'embozo_file_date', 'pos_entry_mode', 'online', 'cvv_ind', 'metodo_identificacion', 'three_ds_status', 'three_ds_flow', 'afiliacion', 'adquirente', 'country', 'cod_respuesta']


In [19]:
schema_preview = get_transaction_schema(SimpleNamespace(context=transaction_agent_context))
print(schema_preview[:4000])

{"row_count": 2964, "column_count": 23, "columns_exposed": ["Regla de fraude", "cb_timestamp", "user_id", "klrid", "transaction_id", "trx_timestamp_mx", "amount_pos", "operador", "mcc_code", "card_type", "product_type", "nombrearchivo", "embozo_file_date", "pos_entry_mode", "online", "cvv_ind", "metodo_identificacion", "three_ds_status", "three_ds_flow", "afiliacion", "adquirente", "country", "cod_respuesta"], "columns": [{"name": "Regla de fraude", "dtype": "float64", "null_count": 2964, "null_rate": 1.0, "sample_values": []}, {"name": "cb_timestamp", "dtype": "str", "null_count": 0, "null_rate": 0.0, "sample_values": ["2026-06-03T02:34:48.341", "2026-05-27T16:57:42.142", "2026-05-28T05:00:28.895", "2026-05-29T03:03:03.254", "2026-05-28T03:51:38.384"]}, {"name": "user_id", "dtype": "str", "null_count": 0, "null_rate": 0.0, "sample_values": ["66bde1be-665c-4378-a040-dd0fcb4acc1a", "1b641e94-8938-4793-a9a0-4c1ec8565450", "596159ed-439f-41fc-a75e-f4c93c88516e", "228d0a48-23a3-453b-b8b4-7

In [20]:
tool_preview = run_transaction_python(
    SimpleNamespace(context=transaction_agent_context),
    """
top_merchants = (
    df.groupby("operador", dropna=False)
      .agg(
          transactions=("transaction_id", "nunique"),
          users=("user_id", "nunique"),
          amount=("amount_pos", "sum"),
      )
      .sort_values(["transactions", "amount"], ascending=False)
      .head(10)
)
top_merchants
    """,
)
print(tool_preview[:5000])

{"status": "ok", "stdout": "", "result": {"type": "DataFrame", "shape": [10, 3], "columns": ["transactions", "users", "amount"], "records": [{"transactions": 478, "users": 421, "amount": 606434.71}, {"transactions": 132, "users": 115, "amount": 28423.69}, {"transactions": 78, "users": 73, "amount": 29421.91}, {"transactions": 78, "users": 70, "amount": 6107.4}, {"transactions": 45, "users": 28, "amount": 3290.0}, {"transactions": 44, "users": 38, "amount": 133881.86}, {"transactions": 41, "users": 10, "amount": 28390.0}, {"transactions": 39, "users": 38, "amount": 6978.56}, {"transactions": 31, "users": 29, "amount": 6909.0}, {"transactions": 27, "users": 26, "amount": 4982.1900000000005}]}}


## Cumulative system prompt experiments

Goal: evaluate how the analyst output changes as more instructions are added to the system prompt.

The flow is:
1. Read `transactional_describer/system_prompt.txt`.
2. Split it by blank-line sections.
3. Build cumulative variants, so variant 1 has only the first section, variant 2 has sections 1-2, and so on.
4. Run every selected variant against the same case dataframe and same analyst request.
5. Append one row per run to the `Prompt Engineering` Google Sheet for side-by-side comparison.

This keeps the case fixed while changing only the system prompt, which makes the experiment easier to compare.


In [21]:
from agents import Runner, set_tracing_disabled
from zoneinfo import ZoneInfo
import hashlib
import time
import traceback

# Experiment configuration. Change these constants only when the source prompt,
# destination sheet, worksheet tab, or service-account path changes.
PROMPT_SOURCE_PATH = Path("transactional_describer/system_prompt.txt")
PROMPT_ENGINEERING_SHEET_ID = "1zJGUX9n36xf3n18t6ieQdGw-1Hd6XP118asDviznUb4"
PROMPT_ENGINEERING_WORKSHEET = "Sheet1"
GOOGLE_SERVICE_ACCOUNT_FILE = Path(os.environ.get("GOOGLE_APPLICATION_CREDENTIALS", ".config/klar-cami-cusi.json"))
SHEET_CELL_CHAR_LIMIT = 49_000

# The notebook logs results/tool calls to Google Sheets, so OpenAI trace export is optional.
# Disable it to avoid non-fatal trace-export 429 warnings during batch prompt runs.
DISABLE_OPENAI_TRACING = True
set_tracing_disabled(DISABLE_OPENAI_TRACING)

# One run writes one row with both evaluation metadata and the model output.
# Keeping raw JSON and the rendered summary makes the sheet useful for quick review
# and later deeper analysis.
SHEET_HEADERS = [
    "experiment_id",
    "run_at_mx",
    "case_id",
    "variant_name",
    "variant_index",
    "prompt_block_count",
    "prompt_chars",
    "prompt_sha256",
    "added_block_preview",
    "status",
    "elapsed_seconds",
    "model",
    "max_turns",
    "case_request",
    "row_count",
    "date_min",
    "date_max",
    "summary",
    "key_patterns",
    "caveats",
    "confidence",
    "raw_output_json",
    "tool_calls_json",
    "error",
    "system_prompt",
]


def clip_sheet_cell(value, limit: int = SHEET_CELL_CHAR_LIMIT) -> str:
    # Google Sheets cells have a hard character limit; clipping prevents one verbose
    # model output or tool trace from failing the whole append operation.
    if value is None:
        return ""
    if isinstance(value, (dict, list)):
        text = json.dumps(value, ensure_ascii=False, default=str)
    elif hasattr(value, "isoformat"):
        text = value.isoformat()
    else:
        text = str(value)
    if len(text) <= limit:
        return text
    return text[: limit - 20] + "\n...[truncated]"


def build_cumulative_prompt_variants(system_prompt_path: Path) -> list[dict]:
    # Each blank-line block becomes a prompt increment. This lets us test whether
    # adding a specific section improves, degrades, or simply changes the output.
    full_prompt = system_prompt_path.read_text(encoding="utf-8").strip()
    blocks = [block.strip() for block in full_prompt.split("\n\n") if block.strip()]
    variants = []
    cumulative_blocks = []

    for idx, block in enumerate(blocks, start=1):
        cumulative_blocks.append(block)
        system_prompt = "\n\n".join(cumulative_blocks)
        variants.append(
            {
                "variant_name": f"system_prompt_{idx:02d}",
                "variant_index": idx,
                "prompt_block_count": idx,
                "added_block": block,
                "added_block_preview": block.splitlines()[0][:240],
                "system_prompt": system_prompt,
                "prompt_chars": len(system_prompt),
                "prompt_sha256": hashlib.sha256(system_prompt.encode("utf-8")).hexdigest()[:16],
            }
        )
    return variants


def open_prompt_engineering_worksheet():
    # Authenticate with the local service account and open the fixed results tab.
    if not GOOGLE_SERVICE_ACCOUNT_FILE.exists():
        raise FileNotFoundError(f"Google service account file not found: {GOOGLE_SERVICE_ACCOUNT_FILE}")
    gc = gspread.service_account(filename=str(GOOGLE_SERVICE_ACCOUNT_FILE))
    spreadsheet = gc.open_by_key(PROMPT_ENGINEERING_SHEET_ID)
    return spreadsheet.worksheet(PROMPT_ENGINEERING_WORKSHEET)


def ensure_prompt_sheet_headers(worksheet, headers: list[str] = SHEET_HEADERS) -> list[str]:
    # First run creates headers. Later runs preserve any existing sheet columns and
    # append missing notebook-defined columns at the end.
    existing_headers = worksheet.row_values(1)
    if not existing_headers:
        worksheet.update(values=[headers], range_name="A1")
        return headers

    missing_headers = [header for header in headers if header not in existing_headers]
    if missing_headers:
        resolved_headers = existing_headers + missing_headers
        worksheet.update(values=[resolved_headers], range_name="A1")
        return resolved_headers
    return existing_headers


def case_metadata(case_df: pd.DataFrame) -> dict:
    # Store lightweight case metadata in every row so results remain interpretable
    # even after the sheet contains runs from multiple cases.
    metadata = {"row_count": len(case_df), "date_min": "", "date_max": ""}
    if "trx_timestamp_mx" in case_df.columns:
        timestamps = pd.to_datetime(case_df["trx_timestamp_mx"], errors="coerce")
        if timestamps.notna().any():
            metadata["date_min"] = timestamps.min().isoformat()
            metadata["date_max"] = timestamps.max().isoformat()
    return metadata


def output_to_dict(output) -> dict:
    # The agent usually returns a Pydantic model; normalize it so sheet logging can
    # handle either structured models or fallback raw outputs.
    if hasattr(output, "model_dump"):
        return output.model_dump()
    if isinstance(output, dict):
        return output
    return {"raw_output": output}


def format_key_patterns(output_dict: dict) -> str:
    # Render key patterns into a readable multiline cell while preserving the raw
    # structured output separately in raw_output_json.
    patterns = output_dict.get("key_patterns") or []
    rendered_patterns = []
    for pattern in patterns:
        title = pattern.get("title", "") if isinstance(pattern, dict) else getattr(pattern, "title", "")
        evidence = pattern.get("evidence", "") if isinstance(pattern, dict) else getattr(pattern, "evidence", "")
        examples = pattern.get("examples", []) if isinstance(pattern, dict) else getattr(pattern, "examples", [])
        example_text = f" Examples: {', '.join(map(str, examples[:5]))}" if examples else ""
        rendered_patterns.append(f"- {title}: {evidence}{example_text}")
    return "\n".join(rendered_patterns)


def append_experiment_row(worksheet, headers: list[str], row: dict):
    # Write values in header order so the notebook can add columns without breaking
    # older rows already present in the sheet.
    values = [clip_sheet_cell(row.get(header, "")) for header in headers]
    worksheet.append_rows([values], value_input_option="USER_ENTERED")


async def run_prompt_experiment(
    *,
    case_df: pd.DataFrame,
    case_id: str,
    analyst_request: str,
    prompt_variants: list[dict],
    model: str | None = None,
    max_turns: int = 8,
    start_variant_index: int = 1,
    max_variants: int | None = None,
) -> pd.DataFrame:
    # Select a contiguous run window. This makes it easy to resume from a later
    # variant or run only the first few variants while debugging.
    selected_variants = [
        variant for variant in prompt_variants
        if variant["variant_index"] >= start_variant_index
    ]
    if max_variants is not None:
        selected_variants = selected_variants[:max_variants]
    if not selected_variants:
        raise ValueError("No prompt variants selected. Check start_variant_index/max_variants.")

    # Open the destination sheet before model calls so credential/header problems
    # fail fast instead of after several paid runs.
    worksheet = open_prompt_engineering_worksheet()
    headers = ensure_prompt_sheet_headers(worksheet)
    metadata = case_metadata(case_df)
    experiment_id = dt.datetime.now(ZoneInfo("America/Mexico_City")).strftime("%Y%m%d-%H%M%S")
    # Prepare one tool context for this case. The same dataframe is reused for every
    # variant so the only experiment variable is the system prompt text.
    transaction_agent_context = prepare_transaction_context(
        case_df,
        important_cols,
        sample_name=f"prompt_experiment_{case_id}",
    )

    rows = []
    for run_position, variant in enumerate(selected_variants, start=1):
        # Reset tool-call tracing before each run so the sheet captures only the
        # Python/tool behavior for the current prompt variant.
        print(
            f"[{run_position}/{len(selected_variants)}] "
            f"Running {variant['variant_name']} ({variant['prompt_chars']} chars)"
        )
        transaction_agent_context.tool_calls = []
        started_at = time.perf_counter()
        run_at_mx = dt.datetime.now(ZoneInfo("America/Mexico_City")).isoformat(timespec="seconds")
        # Start with deterministic metadata, then fill in output or error fields below.
        row = {
            "experiment_id": experiment_id,
            "run_at_mx": run_at_mx,
            "case_id": case_id,
            "variant_name": variant["variant_name"],
            "variant_index": variant["variant_index"],
            "prompt_block_count": variant["prompt_block_count"],
            "prompt_chars": variant["prompt_chars"],
            "prompt_sha256": variant["prompt_sha256"],
            "added_block_preview": variant["added_block_preview"],
            "model": model or "default",
            "max_turns": max_turns,
            "case_request": analyst_request,
            "row_count": metadata["row_count"],
            "date_min": metadata["date_min"],
            "date_max": metadata["date_max"],
            "system_prompt": variant["system_prompt"],
        }

        try:
            # Build a fresh agent per variant because the system instructions are the
            # experiment variable.
            transaction_agent = build_transaction_analyst_agent(
                model=model,
                instructions=variant["system_prompt"],
            )
            result = await Runner.run(
                transaction_agent,
                analyst_request,
                context=transaction_agent_context,
                max_turns=max_turns,
            )
            output_dict = output_to_dict(result.final_output)
            # Store both human-readable fields and raw structured output.
            row.update(
                {
                    "status": "ok",
                    "summary": output_dict.get("summary", ""),
                    "key_patterns": format_key_patterns(output_dict),
                    "caveats": "\n".join(output_dict.get("caveats") or []),
                    "confidence": output_dict.get("confidence", ""),
                    "raw_output_json": output_dict,
                    "tool_calls_json": transaction_agent_context.tool_calls,
                    "error": "",
                }
            )
        except Exception:
            # Log failed variants to the sheet too; failures are useful prompt-test data
            # and make reruns easier to target.
            row.update(
                {
                    "status": "error",
                    "summary": "",
                    "key_patterns": "",
                    "caveats": "",
                    "confidence": "",
                    "raw_output_json": {},
                    "tool_calls_json": transaction_agent_context.tool_calls,
                    "error": traceback.format_exc(limit=8),
                }
            )
        finally:
            row["elapsed_seconds"] = round(time.perf_counter() - started_at, 2)

        # Append immediately after each run so partial experiment results are not lost
        # if a later variant fails or the notebook is interrupted.
        append_experiment_row(worksheet, headers, row)
        rows.append(row)
        print(f"Appended {variant['variant_name']} to Google Sheets with status={row['status']}")

    return pd.DataFrame(rows)


In [22]:
# Build the full prompt ladder once and inspect it before spending model calls.
PROMPT_VARIANTS = build_cumulative_prompt_variants(PROMPT_SOURCE_PATH)

# The preview confirms how many increments will run and which section was added
# at each step. Check this table before running the experiment cell below.
prompt_variants_preview = pd.DataFrame(
    [
        {
            "variant_name": variant["variant_name"],
            "prompt_block_count": variant["prompt_block_count"],
            "prompt_chars": variant["prompt_chars"],
            "prompt_sha256": variant["prompt_sha256"],
            "added_block_preview": variant["added_block_preview"],
        }
        for variant in PROMPT_VARIANTS
    ]
)
prompt_variants_preview


,variant_name,prompt_block_count,prompt_chars,prompt_sha256,added_block_preview
0,system_prompt_01,1,63,8d73496c800b1529,You are a transaction pattern analyst for frau...
1,system_prompt_02,2,316,e3f728f81915387b,Your job is to analyze a transactional datafra...
2,system_prompt_03,3,778,8841fbc07d665bcb,Core objective:
3,system_prompt_04,4,1682,952e1fbbba86ffe5,Hard rules:
4,system_prompt_05,5,3215,9690ff68593790aa,Recommended analysis workflow:
5,system_prompt_06,6,4344,22f24ad9cf66e51c,Key questions to answer when the data supports...
6,system_prompt_07,7,5738,924682c9b33dd8f6,Known fraud-context hypotheses:
7,system_prompt_08,8,6936,7e53327b24f4bae5,Output format:
8,system_prompt_09,9,9130,18023b021a2e031d,Column dictionary:


In [ ]:
# Case configuration: change these values when you want to test a different sample.
# Keep CASE_ID descriptive because it is written to Google Sheets and used for comparisons.
CASE_ID = "chargebacks_platino_31d_16_june2026"
CASE_START_DATE = cb_df["trx_timestamp_mx"].min().strftime("%Y-%m-%d")
# Filter once here so every system-prompt variant analyzes the exact same rows.
CASE_DF = cb_df[pd.to_datetime(cb_df["trx_timestamp_mx"], errors="coerce") >= pd.Timestamp(CASE_START_DATE)].copy()

# User request stays fixed across variants. The experiment should change only the
# system prompt unless you intentionally want to test a different task framing.
CASE_ANALYST_REQUEST = """
Describe the common transaction patterns in this chargeback batch.
Focus on common merchants, repeated user behavior, product/card type differences,
authentication patterns (CVV/3DS/POS entry), countries/acquirers, response codes, amounts, and timing.
Use Python before answering and cite the computed evidence.
""".strip()

# Run controls:
# - MODEL=None uses the Agents SDK default model configured in the environment.
# - START_VARIANT_INDEX lets you resume after a prior partial run.
# - MAX_PROMPT_VARIANTS is useful for a cheap smoke test, for example 1 or 2.
MODEL = None
MAX_TURNS = 8
START_VARIANT_INDEX = 1
MAX_PROMPT_VARIANTS = None

print(f"Case {CASE_ID}: {len(CASE_DF):,} rows")
print(f"Prompt variants selected: {len(PROMPT_VARIANTS) if MAX_PROMPT_VARIANTS is None else MAX_PROMPT_VARIANTS}")

# This is the only cell that calls the model and appends rows to Google Sheets.
prompt_experiment_results = await run_prompt_experiment(
    case_df=CASE_DF,
    case_id=CASE_ID,
    analyst_request=CASE_ANALYST_REQUEST,
    prompt_variants=PROMPT_VARIANTS,
    model=MODEL,
    max_turns=MAX_TURNS,
    start_variant_index=START_VARIANT_INDEX,
    max_variants=MAX_PROMPT_VARIANTS,
)

# Compact local view for quick notebook feedback. The Google Sheet keeps the fuller log.
prompt_experiment_results[[
    "variant_name",
    "prompt_chars",
    "status",
    "elapsed_seconds",
    "summary",
    "confidence",
]]


Case chargebacks_platino_31d_16_june2026: 35 rows
Prompt variants selected: 9
[1/9] Running system_prompt_01 (63 chars)
Appended system_prompt_01 to Google Sheets with status=ok
[2/9] Running system_prompt_02 (316 chars)
Appended system_prompt_02 to Google Sheets with status=ok
[3/9] Running system_prompt_03 (778 chars)


[non-fatal] Tracing client error 429: {
  "error": {
    "message": "You've exceeded the rate limit, please slow down and try again after 60.093360999999994 seconds.",
    "type": "invalid_request_error",
    "param": null,
    "code": "rate_limit_exceeded"
  }
}
[non-fatal] Tracing client error 429: {
  "error": {
    "message": "You've exceeded the rate limit, please slow down and try again after 59.910976000000005 seconds.",
    "type": "invalid_request_error",
    "param": null,
    "code": "rate_limit_exceeded"
  }
}
[non-fatal] Tracing client error 429: {
  "error": {
    "message": "You've exceeded the rate limit, please slow down and try again after 59.998221 seconds.",
    "type": "invalid_request_error",
    "param": null,
    "code": "rate_limit_exceeded"
  }
}


Appended system_prompt_03 to Google Sheets with status=ok
[4/9] Running system_prompt_04 (1682 chars)
Appended system_prompt_04 to Google Sheets with status=ok
[5/9] Running system_prompt_05 (3215 chars)


[non-fatal] Tracing client error 429: {
  "error": {
    "message": "You've exceeded the rate limit, please slow down and try again after 59.960019 seconds.",
    "type": "invalid_request_error",
    "param": null,
    "code": "rate_limit_exceeded"
  }
}


Appended system_prompt_05 to Google Sheets with status=ok
[6/9] Running system_prompt_06 (4344 chars)


[non-fatal] Tracing client error 429: {
  "error": {
    "message": "You've exceeded the rate limit, please slow down and try again after 59.915317 seconds.",
    "type": "invalid_request_error",
    "param": null,
    "code": "rate_limit_exceeded"
  }
}
[non-fatal] Tracing client error 429: {
  "error": {
    "message": "You've exceeded the rate limit, please slow down and try again after 59.930175000000006 seconds.",
    "type": "invalid_request_error",
    "param": null,
    "code": "rate_limit_exceeded"
  }
}


Appended system_prompt_06 to Google Sheets with status=ok
[7/9] Running system_prompt_07 (5738 chars)
Appended system_prompt_07 to Google Sheets with status=ok
[8/9] Running system_prompt_08 (6936 chars)
Appended system_prompt_08 to Google Sheets with status=ok
[9/9] Running system_prompt_09 (9130 chars)
Appended system_prompt_09 to Google Sheets with status=ok


,variant_name,prompt_chars,status,elapsed_seconds,summary,confidence
0,system_prompt_01,63,ok,16.38,This batch is small (35 tx) and heavily CNP / ...,0.84
1,system_prompt_02,316,ok,18.63,This 35-transaction chargeback batch is domina...,0.90
2,system_prompt_03,778,ok,19.56,"35 chargeback transactions, concentrated in la...",0.92
3,system_prompt_04,1682,ok,18.70,This 35-transaction chargeback batch is concen...,0.89
4,system_prompt_05,3215,ok,16.49,This 35-transaction chargeback batch spans 202...,0.83
5,system_prompt_06,4344,ok,21.83,This batch is small (35 chargeback-tagged tran...,0.81
6,system_prompt_07,5738,ok,23.43,This batch is small (35 chargeback transaction...,0.86
7,system_prompt_08,6936,ok,30.83,The batch is small (35 chargeback transactions...,0.78
8,system_prompt_09,9130,ok,22.06,This 35-transaction chargeback batch is concen...,0.90


In [23]:
# SINGLE CASE: full system prompt only
# Keep CASE_ID descriptive because it is written to Google Sheets and used for comparisons.
CASE_ID = "cb_31_days_all"
CASE_START_DATE = cb_df["trx_timestamp_mx"].min().strftime("%Y-%m-%d")
# Filter once here so the full system prompt analyzes the current cb_df rows.
CASE_DF = cb_df[pd.to_datetime(cb_df["trx_timestamp_mx"], errors="coerce") >= pd.Timestamp(CASE_START_DATE)].copy()

# User request stays fixed. The system instructions come from the full prompt below.
CASE_ANALYST_REQUEST = """
Describe the common transaction patterns in this chargeback batch.
Focus on common merchants, repeated user behavior, product/card type differences,
authentication patterns (CVV/3DS/POS entry), countries/acquirers, response codes, amounts, and timing.
Use Python before answering and cite the computed evidence.
""".strip()

# Use the complete system prompt from transactional_describer/system_prompt.txt.
# build_cumulative_prompt_variants stores the full prompt as the final variant.
FULL_PROMPT_VARIANTS = [PROMPT_VARIANTS[-1]]
FULL_PROMPT_VARIANT = FULL_PROMPT_VARIANTS[0]

# Run controls:
# - MODEL=None uses the Agents SDK default model configured in the environment.
# - START_VARIANT_INDEX is pinned to the final prompt variant.
# - MAX_PROMPT_VARIANTS stays None because FULL_PROMPT_VARIANTS already has one item.
MODEL = None
MAX_TURNS = 8
START_VARIANT_INDEX = FULL_PROMPT_VARIANT["variant_index"]
MAX_PROMPT_VARIANTS = None

print(f"Case {CASE_ID}: {len(CASE_DF):,} rows")
print(
    f"Full prompt selected: {FULL_PROMPT_VARIANT['variant_name']} "
    f"({FULL_PROMPT_VARIANT['prompt_chars']:,} chars)"
)
print(f"Prompt variants selected: {len(FULL_PROMPT_VARIANTS)}")

# This is the only cell that calls the model and appends one full-prompt row to Google Sheets.
prompt_experiment_results = await run_prompt_experiment(
    case_df=CASE_DF,
    case_id=CASE_ID,
    analyst_request=CASE_ANALYST_REQUEST,
    prompt_variants=FULL_PROMPT_VARIANTS,
    model=MODEL,
    max_turns=MAX_TURNS,
    start_variant_index=START_VARIANT_INDEX,
    max_variants=MAX_PROMPT_VARIANTS,
)

# 

Case cb_31_days_all: 4,480 rows
Full prompt selected: system_prompt_09 (9,130 chars)
Prompt variants selected: 1
[1/1] Running system_prompt_09 (9130 chars)
Appended system_prompt_09 to Google Sheets with status=ok


In [24]:
prompt_experiment_results

,experiment_id,run_at_mx,case_id,variant_name,variant_index,prompt_block_count,prompt_chars,prompt_sha256,added_block_preview,model,max_turns,case_request,row_count,date_min,date_max,system_prompt,status,summary,key_patterns,caveats,confidence,raw_output_json,tool_calls_json,error,elapsed_seconds
0,20260616-114413,2026-06-16T11:44:13-06:00,cb_31_days_all,system_prompt_09,9,9,9130,18023b021a2e031d,Column dictionary:,default,8,Describe the common transaction patterns in th...,4480,2026-05-16T00:04:45.047000,2026-06-15T23:10:44.658000,You are a transaction pattern analyst for frau...,ok,The batch is highly concentrated in low-to-mid...,- Merchant concentration is led by familiar co...,"cb_timestamp is present on every row, so this ...",0.86,{'summary': 'The batch is highly concentrated ...,"[{'tool': 'get_transaction_schema'}, {'tool': ...",,28.54


In [ ]:
prompt_experiment_results[[
    "variant_name",
    "prompt_chars",
    "status",
    "elapsed_seconds",
    "summary",
    "confidence",
]]


## Multilevel anomaly orchestration

This path is for anomaly discovery rather than general description. It runs a deterministic anomaly screen first, then asks the agent to deep-dive the strongest candidates before producing a ranked report.

In [38]:
ANOMALY_CASE_ID = "cb_31_days_all_anomaly"
ANOMALY_CASE_DF = CASE_DF.copy() 
ANOMALY_ANALYST_REQUEST = """
Find and rank anomalous transaction patterns in this batch.
Do not give me only a general description. First use the anomaly screen, then deep-dive the strongest candidates.
Prioritize concentration, velocity, repeated users/cards, repeated amounts, timing spikes, merchant sequences,
authorization response-code clusters, CVV/3DS/POS-entry anomalies, product/card/country/acquirer/MCC segments,
and low-amount online transactions followed by larger transactions.
For each finding, include computed evidence, examples, why it matters, caveats, and the next analytical check.
""".strip()

anomaly_agent_result = await find_transaction_anomalies(
    ANOMALY_CASE_DF,
    important_cols,
    analyst_request=ANOMALY_ANALYST_REQUEST,
    model=MODEL if "MODEL" in globals() else None,
    max_turns=18,
    sample_name=f"anomaly_{ANOMALY_CASE_ID}",
    timeout_seconds=45,
    max_output_chars=60_000,
)

anomaly_report = anomaly_agent_result.final_output
anomaly_report.model_dump()


{'executive_summary': ['Batch is chargeback-only: all 4,480 rows have cb_timestamp, so chargeback vs non-chargeback comparison is unavailable.',
  'Strongest unusual pattern is a single null merchant bucket: 644 rows (14.4%) and 25.3% of amount, with very high ticket size versus batch median.',
  'A small number of cards/users show elevated velocity; the top card/user has 20 transactions across 14 merchants in 5 days.',
  'Low-amount-to-higher-amount merchant sequencing exists but only as sparse examples; evidence is weak and should be treated as a candidate pattern, not a confirmed abuse rule.'],
 'dataset_overview': '4,480 rows; 3,188 users; 3,394 cards; 1,755 merchants; date range 2026-05-16 to 2026-06-15. Amounts total 3,227,799.33 with mean 720.49, median 200.00, p95 2,777.04, max 43,999.00. Key missingness: operador/afiliacion/adquirente/cod_respuesta each 14.4% null; nombrearchivo/embozo_file_date 54.4% null; country 2.8% null.',
 'anomaly_findings': [{'title': 'Null merchant bu